# Network Anomaly Detection — Section 15 of HTB Module 292

This notebook covers **only** the first network-anomaly section: what a random forest is and how to load/inspect HTB's modified NSL-KDD connection records. Later sections can add preprocessing, training, and evaluation without pretending they have happened here. Run with your `AI` Python environment.

An unusual connection is an **anomaly**, not automatically an intrusion. The dataset's `attack` field gives known labels, but a standard classifier trained on only `normal` rows cannot learn attack classes. Supervised intrusion classification and normal-only anomaly detection are different designs; HTB's introductory wording blends them.

## Random-forest voting, read aloud

A forest contains many decision trees. Each tree trains on a bootstrap sample—rows drawn **with replacement**—and considers random candidate features at each split. For rows A, B, C, D, a four-draw sample might be A, C, C, D: repeated C is allowed and B is missing. For classification, the trees vote. In `ŷ = mode(T₁(x), …, Tᴮ(x))`, say **y hat equals the most common prediction of tree one at x through tree B at x**. `x` is one connection row, `Tᵦ(x)` (tree sub b of x) is tree b's prediction, `B` is the number of trees, and `ŷ` (y hat) is the final predicted class. Votes normal, attack, attack yield attack. This section does **not** fit that forest yet.

## Use the archive linked by HTB

HTB supplies `https://cdn.services-k8s.prod.aws.htb.systems/content/modules/292/KDD_dataset.zip`. Save it as `KDD_dataset.zip` in your Downloads folder, or set `KDD_DATASET_ZIP` to its full local path. The archive stays outside Git. HTB's sample extracts every member into the working directory; we instead read only the expected `KDD+.txt` member directly, so a ZIP cannot place unexpected files in the project. The downloaded archive was verified to contain that one member with a valid CRC.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import pandas as pd

# A full-path environment override keeps the notebook portable across machines.
archive_path = Path(os.environ.get(
    'KDD_DATASET_ZIP', Path.home() / 'Downloads' / 'KDD_dataset.zip')).expanduser()
if not archive_path.is_file():
    raise FileNotFoundError(
        f'Missing {archive_path}. Download KDD_dataset.zip from the HTB lesson link.')
print('Archive:', archive_path.name, '(', archive_path.stat().st_size, 'bytes )')

## Name and load the 43 fields

Each CSV row represents one network **connection record**. The first 41 fields are features, such as `duration`, `protocol_type`, `src_bytes` (source bytes), and `dst_bytes` (destination bytes). `attack` is the known traffic label; `level` is an additional difficulty/metadata value, **not** a second attack label. Giving pandas `names=columns` makes the otherwise headerless file readable. We do not one-hot encode the categorical columns or create a target here; that belongs to later sections.

In [ ]:
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'level'
]
assert len(columns) == 43
with ZipFile(archive_path) as archive:
    if 'KDD+.txt' not in archive.namelist():
        raise ValueError('Expected KDD+.txt member is absent from the HTB ZIP.')
    with archive.open('KDD+.txt') as dataset_file:
        df = pd.read_csv(dataset_file, names=columns)
assert df.shape[1] == len(columns) and len(df) > 0
assert df['attack'].notna().all()
print('Connection records:', len(df), '| columns:', df.shape[1])
print(df.head(3).to_string(index=False))

## First quality and label checks

A label count tells us which traffic types appear most often; it does not yet measure model performance. Missing values and dtypes guide later preprocessing. We inspect small summaries rather than printing every row or converting a large table to an unnecessarily bigger representation. The exact counts belong to this downloaded archive version.

In [ ]:
print('Top attack labels:')
print(df['attack'].value_counts().head(10).to_string())
print('Missing cells:', int(df.isna().sum().sum()))
print('Categorical examples:', df[['protocol_type', 'service', 'flag']].dtypes.to_dict())
print('Level range:', df['level'].min(), 'to', df['level'].max())
assert 'normal' in set(df['attack'])
assert df['attack'].nunique() > 1
assert pd.api.types.is_numeric_dtype(df['level'])

Section 15 ends here: the HTB ZIP is validated, 43 fields are named, and its connection records are inspected. No random forest is trained and no result is called an anomaly score yet. Later sections must define the target, transform categorical fields, split data without leakage, and evaluate before making detection claims.